# NTL F1 — Frozen Qwen3-1.7B-Base decoder fingerprints (T4)

Prospectively fixed: 4 hidden layers, attention-masked pooling (512 tokens), 150-prompt grouped OOF. **Research only**: official 0.702 champion has no honest paired OOF. This notebook does NOT submit to Kaggle or upload raw texts/features to GitHub.

Use a GPU runtime (T4 or better) in Colab. Do not use paid Hugging Face GPU or paid API. Colab ephemeral files are private to this session; only the final aggregate JSON is downloaded. The repo already contains official TRAIN by the owner's previous decision; do not republish those files.

In [ ]:
import os, sys, subprocess
import torch
assert torch.cuda.is_available(), "Change Runtime > Change runtime type > T4 GPU"
assert torch.cuda.get_device_properties(0).total_memory >= 12*1024**3
print("T4_AVAILABLE:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable,"-m","pip","install","-q",
    "transformers==4.57.1","huggingface-hub==0.36.0","accelerate==1.11.0",
    "safetensors==0.6.2","numpy==2.2.6","pandas==2.2.3",
    "scipy==1.15.3","scikit-learn==1.6.1"],check=True)
print("PINNED_DEPENDENCIES_INSTALLED")

In [ ]:
import os, subprocess
ROOT="/content/ntl_f1_code"
assert not os.path.exists(ROOT), "Refuse to overwrite an existing checkout"
subprocess.run(["git","clone","--depth","1","--branch",
    "research/attribution-bottleneck-review-20261106",
    "https://github.com/kolihhan/name-kaggle.git",ROOT],check=True)
os.chdir(ROOT)
print("NTL_F1_CHECKOUT_SHA:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

In [ ]:
import os,sys,subprocess
FEATURES="/content/ntl_f1_qwen3_1p7b_private.npz"
REPORT="/content/ntl_f1_decoder_aggregate_result.json"
assert not os.path.exists(FEATURES) and not os.path.exists(REPORT)
subprocess.run([sys.executable,"-m","scripts.ntl_decoder_probe_extract_colab_20261119",
    "--train","data/train.csv","--folds","artifacts/phase1/folds.csv",
    "--output",FEATURES,"--batch-size","4"],check=True)
print("FROZEN_DECODER_EMBEDDINGS_READY_PRIVATE_COLAB_FILE",os.path.getsize(FEATURES))

In [ ]:
import sys, subprocess,json
subprocess.run([sys.executable,"-m","scripts.ntl_decoder_probe_eval_cpu_20261119",
    "--train","data/train.csv","--folds","artifacts/phase1/folds.csv",
    "--features",FEATURES,"--out",REPORT],check=True)
with open(REPORT) as f: result=json.load(f)
print("C0/C1/C2 DEV_ONLY:",{k:round(v["ba"],6) for k,v in result["results"].items()})
print("Paired comparisons:",json.dumps(result["comparisons"],indent=2))
print("Decision:",result["decision"])
print("NOT AN OFFICIAL KAGGLE SCORE; NO PUBLIC IMPROVEMENT CLAIM")

In [ ]:
from google.colab import files
print("Download aggregate JSON report ONLY; do NOT upload private features or competition texts to public GitHub.")
files.download(REPORT)

## Interpretation

**C1−C0:** Does a pinned frozen decoder representation beat a matched lexical readout on historical, prompt-disjoint grouped development folds?

**C2−C1:** Does a nuisance direction estimated exclusively from training prompt centroids improve further or erase real model signals?

Strong previous own-LoRA24 grouped OOF 0.5468 is historical context only and is not automatically a same-row eligible control. The attributed 0.702 Kaggle Public model has **no honest OOF for a paired comparison**. Do not estimate hidden gains from these results.